# UTR-2 transient catalogue

Query the transient catalogue of the **UTR-2 Pulsar/Transient Survey of the
Northern Sky (UTPSNS)** and render it over the 20 MHz galactic background map.

The service returns:

* a table of every transient matching the query,
* a sky map with the selected transients marked,
* distribution histograms of galactic latitude, SNR, flux and dispersion measure.

Leave `radius` at its default to search the whole sky; set `RA`, `DEC` and a
smaller `radius` to perform a cone search around a position or a named source.


In [ ]:
# oda:version "v0.1.0"
# oda:reference "https://doi.org/10.1007/s10509-018-3433-8"

src_name = "Cas A"  # http://odahub.io/ontology#AstrophysicalObject ; oda:label "Source name"

RA = 350.85  # http://odahub.io/ontology#PointOfInterestRA ; oda:label "RA"
DEC = 58.815  # http://odahub.io/ontology#PointOfInterestDEC ; oda:label "Dec"

radius = 180.0  # http://odahub.io/ontology#AngleDegrees ; oda:lower_limit 0. ;
                # oda:upper_limit 180. ; oda:label "Search radius" ;
                # oda:description "Cone search radius in degrees. 180 searches the whole sky."

snr_threshold = 8.0  # http://odahub.io/ontology#Float ; oda:lower_limit 0. ;
                     # oda:label "Minimum SNR" ; oda:group "Selection" ;
                     # oda:description "Keep transients with corrected SNR strictly above this value."

dm_min = 0.0  # http://odahub.io/ontology#Float ; oda:lower_limit 0. ;
              # oda:label "Minimum DM" ; oda:group "Selection" ;
              # oda:description "Lowest corrected dispersion measure, pc/cm3."

dm_max = 100.0  # http://odahub.io/ontology#Float ; oda:lower_limit 0. ;
                # oda:label "Maximum DM" ; oda:group "Selection" ;
                # oda:description "Highest corrected dispersion measure, pc/cm3."


## Setup

Locate the repository, select a head-less matplotlib backend and import the shared analysis modules that the desktop application also uses.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib

# The notebook is executed head-lessly by papermill: there is no display, so the
# Agg backend must be selected before any figure is created. (The desktop
# application in the repository root uses TkAgg instead.)
matplotlib.use("Agg")

from matplotlib.figure import Figure


REPO_MARKER = Path("src") / "query.py"


def ensure_src_importable() -> None:
    """Make the shared `src` package importable, however the notebook is run.

    Normally the project is pip-installed (`pip install -e .`), in which case
    `src` is already on the path and nothing happens here. The fallback covers
    running straight from a git checkout: papermill executes with the working
    directory of whoever launched it, not the notebook's own folder, so we walk
    upwards looking for the package and allow an explicit override.
    """
    try:
        import src.query  # noqa: F401
        return
    except ImportError:
        pass

    candidates = []
    override = os.environ.get("UTR2_REPO_ROOT")
    if override:
        candidates.append(Path(override).resolve())
    start = Path(os.getcwd()).resolve()
    candidates.extend([start, *start.parents])

    for candidate in candidates:
        if (candidate / REPO_MARKER).exists():
            sys.path.insert(0, str(candidate))
            return

    raise RuntimeError(
        f"Cannot import the `src` package and could not find {REPO_MARKER} by "
        f"searching upwards from {start}. Install the project with "
        "`pip install -e .` from the repository root, or set the UTR2_REPO_ROOT "
        "environment variable to point at it."
    )


ensure_src_importable()

from src.coordinates.transforms import equatorial_to_galactic
from src.data.transient_loader import load_transients
from src.maps.jpeg_map import JpegBackgroundMap
from src.plots.histograms import HistogramPanel
from src.plots.skymap import render_sky_map
from src.products import query_to_table
from src.query import FULL_SKY_RADIUS_DEG, filter_catalog

import src

# The catalogue and the background image live next to the package, so derive
# their location from the package rather than from the working directory.
REPO_ROOT = Path(src.__file__).resolve().parent.parent
CSV_PATH = REPO_ROOT / "Data" / "Tr_380_Flux.csv"
MAP_PATH = REPO_ROOT / "assets" / "GalBackgr20MHz-1.jpg"

for _required in (CSV_PATH, MAP_PATH):
    if not _required.exists():
        raise RuntimeError(
            f"Missing data file {_required}. If the project was installed "
            "non-editably, reinstall it with `pip install -e .` so the Data/ "
            "and assets/ folders stay alongside the package."
        )
print(f"Repository root: {REPO_ROOT}")

# Reproduces the desktop application's fixed declination ceiling. The catalogue
# tops out at Dec = +74.5, so this currently excludes nothing; it is kept for
# fidelity with the original IDL selection.
MAX_DEC_DEG = 75.0


## Load the catalogue

Read the CSV and attach galactic coordinates.

In [ ]:
catalog = load_transients(CSV_PATH)
gl, gb = equatorial_to_galactic(catalog.ra, catalog.dec)
catalog = catalog.with_galactic(gl, gb)

print(f"Loaded {len(catalog)} transients from {CSV_PATH.name}")


## Apply the query

In [ ]:
cone_requested = radius is not None and radius < FULL_SKY_RADIUS_DEG

# Raises NoTransientsFound with an explanatory message if nothing matches;
# MMODA shows that message to the user.
result = filter_catalog(
    catalog,
    ra_deg=RA if cone_requested else None,
    dec_deg=DEC if cone_requested else None,
    radius_deg=radius,
    snr_min=snr_threshold,
    dm_min=dm_min,
    dm_max=dm_max,
    dec_max=MAX_DEC_DEG,
)

print(f"{len(result)} of {result.n_total} transients match the query.")


## Build the table product

In [ ]:
request_meta = {
    "SRCNAME": src_name or "",
    "QRA": RA if cone_requested else None,
    "QDEC": DEC if cone_requested else None,
    "QRADIUS": radius if cone_requested else None,
    "SNRMIN": snr_threshold,
    "DMMIN": dm_min,
    "DMMAX": dm_max,
}
# FITS headers reject None, so drop the keys that do not apply.
request_meta = {k: v for k, v in request_meta.items() if v is not None}

transients = query_to_table(result, meta=request_meta)
transients


## Render the figures

In [ ]:
cone_center = (RA, DEC) if cone_requested else None
title = "UTR-2 transients on the 20 MHz galactic background"
if cone_requested:
    label = src_name.strip() if src_name and src_name.strip() else f"RA={RA:.3f}, Dec={DEC:.3f}"
    title = f"UTR-2 transients within {radius:g} deg of {label}"

sky_figure = render_sky_map(
    result.catalog,
    JpegBackgroundMap(MAP_PATH),
    cone_center=cone_center,
    cone_radius_deg=radius if cone_requested else None,
    title=title,
)
sky_map_fname = "utr2_sky_map.png"
sky_figure.savefig(sky_map_fname, dpi=110, bbox_inches="tight")

hist_figure = Figure(figsize=(4.2, 9.0), layout="constrained")
HistogramPanel(result.catalog, snr_threshold=snr_threshold, figure=hist_figure)
histograms_fname = "utr2_histograms.png"
hist_figure.savefig(histograms_fname, dpi=110, bbox_inches="tight")

print(f"Wrote {sky_map_fname} and {histograms_fname}")


## Summary comment

In [ ]:
summary = (
    f"{len(result)} of {result.n_total} UTR-2 transients match this query "
    f"(SNR_corr > {snr_threshold}, {dm_min} <= DM_corr <= {dm_max}"
)
summary += (
    f", within {radius:g} deg of RA={RA:.4f}, Dec={DEC:.4f})."
    if cone_requested
    else ", whole sky)."
)
summary += (
    " Note: the catalogue records no absolute observation epoch, so this "
    "service cannot be filtered by time."
)
print(summary)


## Outputs

In [ ]:
transient_table = transients  # http://odahub.io/ontology#ODAAstropyTable
sky_map = sky_map_fname  # http://odahub.io/ontology#ODAPictureProduct
histograms = histograms_fname  # http://odahub.io/ontology#ODAPictureProduct
query_summary = summary  # http://odahub.io/ontology#WorkflowResultComment
